# CoherenceBench-IN — Environment Setup

Run this notebook once on Google Colab to set up all dependencies.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# ─── 1. Install Dependencies ───────────────────────────────────────
!pip install -q \
    torch \
    transformers>=4.40.0 \
    datasets>=2.19.0 \
    accelerate>=0.30.0 \
    bitsandbytes>=0.43.0 \
    sentencepiece \
    tiktoken \
    spacy>=3.7.0 \
    evaluate \
    bert-score \
    pandas \
    numpy \
    jsonlines \
    tqdm \
    matplotlib \
    seaborn \
    python-dotenv

# Download spaCy English model
!python -m spacy download en_core_web_sm

print("\n✅ All packages installed.")

In [ ]:
# ─── 2. Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/coherencebench-in'
for subdir in ['data/raw', 'data/processed', 'data/benchmark', 'results', 'checkpoints']:
    os.makedirs(f'{PROJECT_DIR}/{subdir}', exist_ok=True)

print(f'✅ Project directory ready at: {PROJECT_DIR}')

In [ ]:
# ─── 3. Verify GPU ────────────────────────────────────────────────
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'VRAM:            {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ─── 4. Verify Key Imports ─────────────────────────────────────────
import transformers
import datasets
import tiktoken
import spacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f'transformers:  {transformers.__version__}')
print(f'datasets:      {datasets.__version__}')
print(f'pandas:        {pd.__version__}')
print(f'numpy:         {np.__version__}')

# Load spaCy model
nlp = spacy.load('en_core_web_sm')
doc = nlp('Dr. Sharma works at AIIMS in New Delhi since 2015.')
entities = [(ent.text, ent.label_) for ent in doc.ents]
print(f'\nspaCy NER test: {entities}')
print('\n✅ All imports working.')

In [ ]:
# ─── 5. Test Model Loading (4-bit Quantized) ──────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'  # smallest model — quick test

print(f'Loading {MODEL_NAME} in 4-bit...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map='auto'
)

# Quick inference test
prompt = 'The capital of France is'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=10)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f'\nTest prompt: "{prompt}"')
print(f'Model output: "{response}"')
print('\n✅ Model loading and inference working.')

# Clean up to free VRAM
del model, tokenizer
torch.cuda.empty_cache()

In [ ]:
# ─── 6. Test Wikipedia Dataset Loading ─────────────────────────────
from datasets import load_dataset

print('Loading English Wikipedia sample (streaming)...')
wiki = load_dataset('wikipedia', '20220301.en', split='train', streaming=True)

# Grab first 5 articles
enc = tiktoken.get_encoding('cl100k_base')
for i, article in enumerate(wiki):
    if i >= 5:
        break
    tokens = len(enc.encode(article['text']))
    print(f'  Article {i+1}: "{article["title"]}" — {tokens:,} tokens')

print('\n✅ Wikipedia dataset accessible.')

In [ ]:
# ─── 7. Tokenizer Benchmark ───────────────────────────────────────
# Verify tiktoken token counting is consistent
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')

test_texts = [
    'Short sentence.',
    'Dr. Sharma, a renowned cardiologist at the All India Institute of Medical Sciences (AIIMS), published her groundbreaking research on cardiac arrhythmias in the New England Journal of Medicine in 2019.',
    'The expedition began on March 15, 1924, and lasted approximately 47 days before the team reached the summit on May 1.',
]

for text in test_texts:
    n_tokens = len(enc.encode(text))
    print(f'{n_tokens:4d} tokens | "{text[:80]}..."' if len(text) > 80 else f'{n_tokens:4d} tokens | "{text}"')

print('\n✅ Tokenizer working. Use tiktoken cl100k_base for standardized token counts.')

---
## Setup Complete!

All dependencies installed and verified:
- ✅ GPU available (T4)
- ✅ Google Drive mounted
- ✅ 4-bit model loading works
- ✅ Wikipedia dataset accessible
- ✅ spaCy NER working
- ✅ Tokenizer ready

**Next step:** Phase 1 — Literature Review. Start reading RULER (Hsieh et al., 2024).